CSV

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

class LogParser:
    """
    Handles parsing of Linux log entries with timestamp and service extraction
    Supports standard syslog format: Month Day Time Hostname Service: Message
    """

    def __init__(self):
        self.log_pattern = re.compile(
            r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})\s+(\S+)\s+([^:]+):\s*(.*)'
        )

    def normalize_service_name(self, service):
        """
        Normalize service names by removing PID numbers
        Examples:
        - sshd(pam_unix)[19431] -> sshd(pam_unix)
        - ftpd[23152] -> ftpd
        - sdpd[1681] -> sdpd
        - kernel -> kernel (unchanged)
        """
        if not service:
            return service

        # Remove PID numbers in square brackets: [1234]
        normalized = re.sub(r'\[\d+\]', '', service)

        # Clean up any trailing whitespace
        normalized = normalized.strip()

        return normalized

    def parse_log_entry(self, log_line, current_year=2024):
        """Parse a single log entry into structured components"""
        log_line = log_line.strip()
        if not log_line:
            return None

        match = self.log_pattern.match(log_line)

        if not match:
            # Fallback for logs that don't match expected format
            return {
                'timestamp': None,
                'hostname': 'unknown',
                'service': 'unknown',
                'message': log_line,
                'raw_line': log_line
            }

        timestamp_str, hostname, service, message = match.groups()

        # Parse timestamp (add current year since syslog doesn't include year)
        try:
            timestamp = datetime.strptime(f"{current_year} {timestamp_str}", "%Y %b %d %H:%M:%S")
        except ValueError:
            timestamp = None

        # Normalize the service name to remove PIDs
        normalized_service = self.normalize_service_name(service.strip())

        return {
            'timestamp': timestamp,
            'hostname': hostname,
            'service': normalized_service,
            'message': message.strip(),
            'raw_line': log_line
        }

    def parse_log_file(self, file_path, current_year=2024):
        """Parse entire log file and return structured data"""
        print(f"📂 Reading log file: {file_path}")

        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.readlines()
            print(f"✅ Successfully read {len(lines)} lines")
        except FileNotFoundError:
            print(f"❌ Error: File '{file_path}' not found")
            return []
        except Exception as e:
            print(f"❌ Error reading file: {e}")
            return []

        parsed_logs = []
        processed_count = 0

        for i, line in enumerate(lines):
            if i % 5000 == 0 and i > 0:
                print(f"📝 Processed {i}/{len(lines)} lines...")

            parsed_entry = self.parse_log_entry(line, current_year)
            if parsed_entry:
                parsed_logs.append(parsed_entry)
                processed_count += 1

        print(f"✅ Parsing complete! Extracted {processed_count} valid log entries")
        return parsed_logs

class MessageNormalizer:
    """
    Handles normalization of log messages for pattern detection
    Replaces variable parts (PIDs, IPs, hex addresses) with placeholders
    """

    @staticmethod
    def normalize_message(message):
        """Normalize message by replacing variable parts with placeholders"""
        normalized = message

        # Replace process IDs and task IDs
        normalized = re.sub(r'\[\d+\]', '[PID]', normalized)
        normalized = re.sub(r'process \d+', 'process <PID>', normalized)
        normalized = re.sub(r'(?i)pid\s*=\s*\d+', 'pid=<PID>', normalized)
        normalized = re.sub(r'task \d+', 'task <PID>', normalized)

        # Replace network addresses and hex values
        normalized = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', normalized)
        normalized = re.sub(r'0x[a-fA-F0-9]+', '<HEX>', normalized)

        # Replace standalone numbers
        normalized = re.sub(r'\b\d+\b', '<NUM>', normalized)

        return normalized

class ImprovedFaultClassifier:
    """
    Enhanced fault classification with comprehensive pattern matching
    Includes detection for:
    - Kernel panics and crashes
    - Memory faults (OOM, segfaults)
    - Authentication failures
    - Service failures
    - Stack traces and kernel function calls
    - Process exit failures
    """

    def __init__(self):
        # Severity-weighted fault indicator words
        self.fault_words = {
            'critical': ['panic', 'fatal', 'crash', 'died', 'killed', 'emergency', 'oops'],
            'high': ['error', 'fail', 'failed', 'failure', 'fault', 'timeout', 'denied', 'unable', 'refused', 'exited'],
            'medium': ['warning', 'warn', 'slow', 'retry', 'delayed'],
        }

        # Technical fault patterns (regex)
        self.tech_patterns = [
            # Memory and system faults
            r'out of memory|oom|segfault',
            r'kernel panic|oops:|call trace',
            r'hardware error|disk.*error',

            # Authentication and access faults
            r'authentication.*fail|access.*denied',

            # Service management faults
            r'shutdown failed',
            r'shutdown.*failed',
            r'failed.*shutdown',
            r'stopping.*failed',
            r'failed.*stop',
            r'.*:\s*failed\s*$',
            r'service.*failed',
            r'daemon.*failed',
            r'start.*failed|failed.*start',

            # Process exit failures (NEW)
            r'exited abnormally',                     # General abnormal exits
            r'exited abnormally with \[\d+\]',        # Specific: "exited abnormally with [1]"
            r'exited with.*code\s+[1-9]\d*',          # "exited with code 1", "exited with exit code 2"
            r'terminated abnormally',                 # Process termination issues
            r'abnormal.*termination',                 # Alternative phrasing
            r'process.*exit.*[1-9]\d*',               # "process exit 1", "process exit code 2"

            # Stack traces and kernel function calls
            r'\[<[a-fA-F0-9]+>\]',                    # Stack trace addresses like [<c0142d8c>]
            r'do_exit\+',                             # Kernel function do_exit
            r'page_fault\+',                          # Page fault handlers
            r'do_page_fault\+',                       # Page fault processing
            r'panic\+',                               # Panic function calls
            r'oops_end\+',                            # Oops handling
            r'handle_mm_fault\+',                     # Memory management faults
            r'sysrq_handle_crash\+',                  # System request crash handling
            r'error_code\+',                          # Error code handling
            r'.*_fault\+0x[a-fA-F0-9]+',             # Any fault function with hex offset
            r'.*_exit\+0x[a-fA-F0-9]+',              # Any exit function with hex offset
            r'.*panic\+0x[a-fA-F0-9]+',              # Any panic function with hex offset
            r'.*oops.*\+0x[a-fA-F0-9]+',             # Any oops function with hex offset
            r'.*error.*\+0x[a-fA-F0-9]+',            # Any error function with hex offset
            r'\+0x[a-fA-F0-9]+/0x[a-fA-F0-9]+',      # Function offset pattern like +0x3fc/0x430
        ]

        # Patterns indicating successful operations (NOT faults)
        self.positive_patterns = [
            r'\[\s*OK\s*\]',
            r'\033\[.*?m.*OK',
            r'started successfully',
            r'completed successfully',
            r'restart\.',
            r'available\.',
            r'exited normally',                       # Normal exits are not faults
            r'exited.*with.*code\s+0',               # Exit code 0 is success
        ]

    def analyze_message(self, message):
        """
        Analyze message and return fault probability (0.0 to 1.0)
        Higher values indicate higher likelihood of being a fault
        """
        msg_lower = message.lower()

        # Step 1: Check for success patterns first (these are NOT faults)
        for positive_pattern in self.positive_patterns:
            if re.search(positive_pattern, message, re.IGNORECASE):
                return 0.05  # Very low fault probability

        # Step 2: High-priority explicit checks

        # Stack traces are almost always faults
        if re.search(r'\[<[a-fA-F0-9]+>\]', message):
            return 0.95  # Very high fault probability for stack traces

        # Abnormal exits are high-priority faults
        if re.search(r'exited abnormally', msg_lower):
            return 0.9   # Very high fault probability for abnormal exits

        # Explicit shutdown failures
        if 'shutdown failed' in msg_lower:
            return 0.95

        # Messages ending with "failed"
        if msg_lower.strip().endswith('failed'):
            return 0.9

        # Step 3: Fault word scoring
        fault_score = 0
        for severity, words in self.fault_words.items():
            severity_weight = {'critical': 1.0, 'high': 0.8, 'medium': 0.5}[severity]
            for word in words:
                if re.search(rf'\b{word}\b', msg_lower):
                    fault_score += severity_weight

        # Step 4: Technical pattern matching
        for pattern in self.tech_patterns:
            if re.search(pattern, msg_lower, re.IGNORECASE):
                fault_score += 0.8

        # Step 5: Convert score to probability
        fault_prob = min(1.0, fault_score / 1.5)
        return fault_prob


class LogClusterer:
    """
    Handles clustering of log messages for pattern discovery
    Groups similar messages together to identify rare fault patterns
    """

    def __init__(self, max_features=500):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            stop_words='english',
            ngram_range=(1, 2),
            max_df=0.8,
            min_df=0.01
        )
        self.normalizer = MessageNormalizer()

    def cluster_messages(self, messages, n_clusters=10):
        """Cluster messages based on their normalized content"""
        print(f"🎯 Clustering {len(messages)} messages...")

        # Normalize messages for better clustering
        normalized_messages = [self.normalizer.normalize_message(msg) for msg in messages]

        # Vectorize messages
        features = self.vectorizer.fit_transform(normalized_messages)

        # Adjust cluster count based on data size
        actual_clusters = min(n_clusters, max(2, len(messages) // 10))

        # Perform clustering
        kmeans = KMeans(n_clusters=actual_clusters, random_state=42, n_init=10)
        clusters = kmeans.fit_predict(features)

        print(f"✅ Created {actual_clusters} clusters")
        return clusters

class FaultTypeClassifier:
    """
    Determines the specific type of fault based on message content
    Categories: memory_fault, kernel_fault, security_fault, service_fault, hardware_fault, process_fault, general_fault
    """

    @staticmethod
    def classify_fault_type(message, is_fault):
        """Classify the type of fault based on message content"""
        if not is_fault:
            return 'normal'

        msg_lower = message.lower()

        # Process exit faults (NEW - high priority)
        if re.search(r'exited abnormally|terminated abnormally|abnormal.*termination', msg_lower):
            return 'process_fault'

        # Stack trace / Kernel function calls (high priority)
        elif re.search(r'\[<[a-fA-F0-9]+>\]|.*\+0x[a-fA-F0-9]+', message):
            return 'kernel_fault'

        # Memory-related faults
        elif any(word in msg_lower for word in ['memory', 'oom', 'vm:', 'out of memory', 'mm_fault', 'segfault']):
            return 'memory_fault'

        # Kernel-related faults
        elif any(word in msg_lower for word in ['panic', 'crash', 'kernel', 'oops', 'do_exit', 'page_fault']):
            return 'kernel_fault'

        # Security-related faults
        elif any(word in msg_lower for word in ['auth', 'login', 'failure', 'denied', 'unauthorized']):
            return 'security_fault'

        # Service-related faults
        elif any(word in msg_lower for word in ['shutdown', 'stop', 'service', 'daemon']):
            return 'service_fault'

        # Hardware-related faults
        elif any(word in msg_lower for word in ['hardware', 'disk', 'thermal', 'machine check']):
            return 'hardware_fault'

        # General fault
        else:
            return 'general_fault'


class LogClassificationPipeline:
    """
    Main pipeline for processing and classifying Linux system logs
    Orchestrates parsing, linguistic analysis, clustering, and classification
    """

    def __init__(self):
        self.parser = LogParser()
        self.normalizer = MessageNormalizer()
        self.fault_classifier = ImprovedFaultClassifier()
        self.clusterer = LogClusterer()
        self.fault_type_classifier = FaultTypeClassifier()

    def process_log_file(self, file_path, current_year=2024):
        """Process a Linux log file and return classified results"""
        print("🚀 Starting Linux log classification pipeline...")
        print("=" * 60)

        # Step 1: Parse log file
        parsed_logs = self.parser.parse_log_file(file_path, current_year)
        if not parsed_logs:
            print("❌ No logs to process")
            return []

        # Extract messages for analysis
        messages = [log['message'] for log in parsed_logs]

        # Step 2: Linguistic fault analysis
        print("\n🔤 Performing linguistic fault analysis...")
        linguistic_scores = []
        for i, msg in enumerate(messages):
            if i % 5000 == 0 and i > 0:
                print(f"  Analyzed {i}/{len(messages)} messages...")

            score = self.fault_classifier.analyze_message(msg)
            linguistic_scores.append(score)

        print(f"✅ Linguistic analysis complete!")

        # Step 3: Clustering analysis for pattern discovery
        print("\n🎯 Performing clustering analysis...")
        clusters = self.clusterer.cluster_messages(messages)

        # Step 4: Analyze clusters to determine fault patterns
        cluster_fault_probs = self._analyze_clusters(clusters, linguistic_scores)

        # Step 5: Generate final ensemble classifications
        print("\n⚡ Generating final ensemble classifications...")
        results = []

        for i, parsed_log in enumerate(parsed_logs):
            linguistic_prob = linguistic_scores[i]
            cluster_prob = cluster_fault_probs[clusters[i]]

            # Ensemble decision: combine linguistic and clustering analysis
            final_prob = (linguistic_prob * 0.6) + (cluster_prob * 0.4)
            is_fault = final_prob > 0.5
            confidence = abs(final_prob - 0.5) * 2  # Distance from decision boundary

            # Determine specific fault type
            fault_type = self.fault_type_classifier.classify_fault_type(
                parsed_log['message'], is_fault
            )

            # Normalize message for pattern analysis
            normalized_msg = self.normalizer.normalize_message(parsed_log['message'])

            result = {
                'timestamp': parsed_log['timestamp'],
                'service': parsed_log['service'],
                'message': parsed_log['message'],
                'normalized_message': normalized_msg,
                'is_fault': is_fault,
                'fault_type': fault_type,
                'confidence': confidence,
                'fault_probability': final_prob,
                'cluster_id': clusters[i]
            }

            results.append(result)

        print(f"✅ Classification complete! Processed {len(results)} log entries")
        return results

    def _analyze_clusters(self, clusters, linguistic_scores):
        """
        Analyze clusters to determine fault probabilities
        Small clusters with high linguistic scores are likely faults
        """
        cluster_fault_probs = {}

        for cluster_id in set(clusters):
            cluster_indices = [i for i, c in enumerate(clusters) if c == cluster_id]
            cluster_linguistic_scores = [linguistic_scores[i] for i in cluster_indices]

            # Calculate cluster statistics
            avg_linguistic = np.mean(cluster_linguistic_scores)
            cluster_size = len(cluster_indices)

            # Size factor: smaller clusters are more likely to be faults (rare events)
            size_factor = min(1.0, 20 / cluster_size)

            # Combine linguistic and size factors
            cluster_fault_prob = (avg_linguistic * 0.7) + (size_factor * 0.3)
            cluster_fault_probs[cluster_id] = cluster_fault_prob

        return cluster_fault_probs

    def save_results_to_csv(self, results, filename="classified_logs.csv"):
        """Save results to CSV with clean column selection"""
        if not results:
            print("❌ No results to save")
            return None

        df = pd.DataFrame(results)

        # Select columns for CSV output (exclude confidence and fault_probability as requested)
        csv_columns = [
            'timestamp', 'service', 'message', 'normalized_message',
            'is_fault', 'fault_type', 'cluster_id'
        ]

        csv_df = df[csv_columns].copy()

        # Convert timestamp to string for CSV compatibility
        if 'timestamp' in csv_df.columns:
            csv_df['timestamp'] = csv_df['timestamp'].astype(str)

        # Save to CSV
        csv_df.to_csv(filename, index=False)
        print(f"💾 Results saved to: {filename}")

        return csv_df

    def print_analysis_summary(self, results):
        """Print comprehensive analysis summary"""
        if not results:
            return

        df = pd.DataFrame(results)
        total = len(df)
        faults = df['is_fault'].sum()

        print(f"\n{'='*60}")
        print(f"📊 ANALYSIS SUMMARY")
        print(f"{'='*60}")

        # Overall statistics
        print(f"📈 OVERALL STATISTICS:")
        print(f"  Total logs processed: {total:,}")
        print(f"  Faults detected: {faults:,} ({faults/total*100:.1f}%)")
        print(f"  Normal operations: {total-faults:,} ({(total-faults)/total*100:.1f}%)")

        # Fault type distribution
        if faults > 0:
            print(f"\n🔍 FAULT TYPE DISTRIBUTION:")
            fault_types = df[df['is_fault']]['fault_type'].value_counts()
            for fault_type, count in fault_types.items():
                percentage = (count / faults) * 100
                print(f"  {fault_type}: {count:,} ({percentage:.1f}%)")

        # Service distribution
        print(f"\n🔧 TOP SERVICES:")
        service_counts = df['service'].value_counts().head(10)
        for service, count in service_counts.items():
            percentage = (count / total) * 100
            print(f"  {service}: {count:,} ({percentage:.1f}%)")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Configuration
    LOG_FILE_PATH = "Linux.log"  # Change this to your log file path
    OUTPUT_CSV = "classified_logs.csv"
    CURRENT_YEAR = 2024  # Adjust based on your log year

    # Initialize and run the pipeline
    pipeline = LogClassificationPipeline()

    # Process the log file
    results = pipeline.process_log_file(LOG_FILE_PATH, CURRENT_YEAR)

    if results:
        # Save results to CSV
        csv_df = pipeline.save_results_to_csv(results, OUTPUT_CSV)

        # Print comprehensive analysis
        pipeline.print_analysis_summary(results)

        print(f"\n🎉 PROJECT VANGUARD LOG PROCESSING COMPLETE!")

    else:
        print("❌ No results generated. Please check your log file.")


🚀 Starting Linux log classification pipeline...
📂 Reading log file: /content/Linux (1).log
✅ Successfully read 25567 lines
📝 Processed 5000/25567 lines...
📝 Processed 10000/25567 lines...
📝 Processed 15000/25567 lines...
📝 Processed 20000/25567 lines...
📝 Processed 25000/25567 lines...
✅ Parsing complete! Extracted 25567 valid log entries

🔤 Performing linguistic fault analysis...
  Analyzed 5000/25567 messages...
  Analyzed 10000/25567 messages...
  Analyzed 15000/25567 messages...
  Analyzed 20000/25567 messages...
  Analyzed 25000/25567 messages...
✅ Linguistic analysis complete!

🎯 Performing clustering analysis...
🎯 Clustering 25567 messages...
✅ Created 10 clusters

⚡ Generating final ensemble classifications...
✅ Classification complete! Processed 25567 log entries
💾 Results saved to: classified_logs.csv

📊 ANALYSIS SUMMARY
📈 OVERALL STATISTICS:
  Total logs processed: 25,567
  Faults detected: 15,935 (62.3%)
  Normal operations: 9,632 (37.7%)

🔍 FAULT TYPE DISTRIBUTION:
  memor

For Realtime logs


In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

class RealTimeLogProcessor:
    """
    Optimized real-time log processor for batch processing without file I/O
    Fast classification with direct output printing
    """

    def __init__(self):
        self.log_pattern = re.compile(
            r'(\w{3}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})\s+(\S+)\s+([^:]+):\s*(.*)'
        )

        # Fault classification patterns
        self.fault_words = {
            'critical': ['panic', 'fatal', 'crash', 'died', 'killed', 'emergency', 'oops'],
            'high': ['error', 'fail', 'failed', 'failure', 'fault', 'timeout', 'denied', 'unable', 'refused', 'exited'],
            'medium': ['warning', 'warn', 'slow', 'retry', 'delayed'],
        }

        self.tech_patterns = [
            r'out of memory|oom|segfault',
            r'kernel panic|oops:|call trace',
            r'hardware error|disk.*error',
            r'authentication.*fail|access.*denied',
            r'shutdown failed|shutdown.*failed|failed.*shutdown',
            r'stopping.*failed|failed.*stop|.*:\s*failed\s*$',
            r'service.*failed|daemon.*failed|start.*failed|failed.*start',
            r'exited abnormally|exited abnormally with \[\d+\]|exited with.*code\s+[1-9]\d*',
            r'terminated abnormally|abnormal.*termination|process.*exit.*[1-9]\d*',
            r'\[<[a-fA-F0-9]+>\]|.*\+0x[a-fA-F0-9]+',
            r'do_exit\+|page_fault\+|do_page_fault\+|panic\+|oops_end\+',
            r'handle_mm_fault\+|sysrq_handle_crash\+|error_code\+',
        ]

        self.positive_patterns = [
            r'\[\s*OK\s*\]|started successfully|completed successfully',
            r'restart\.|available\.|exited normally|exited.*with.*code\s+0'
        ]

    def normalize_service_name(self, service):
        """Remove PID numbers from service names"""
        return re.sub(r'\[\d+\]', '', service).strip()

    def normalize_message(self, message):
        """Normalize message for pattern detection"""
        normalized = message
        normalized = re.sub(r'\[\d+\]', '[PID]', normalized)
        normalized = re.sub(r'process \d+', 'process <PID>', normalized)
        normalized = re.sub(r'(?i)pid\s*=\s*\d+', 'pid=<PID>', normalized)
        normalized = re.sub(r'task \d+', 'task <PID>', normalized)
        normalized = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', normalized)
        normalized = re.sub(r'0x[a-fA-F0-9]+', '<HEX>', normalized)
        normalized = re.sub(r'\b\d+\b', '<NUM>', normalized)
        return normalized

    def analyze_fault_probability(self, message):
        """Fast fault probability analysis"""
        msg_lower = message.lower()

        # Quick positive pattern check
        for pattern in self.positive_patterns:
            if re.search(pattern, message, re.IGNORECASE):
                return 0.05

        # High-priority explicit checks
        if re.search(r'\[<[a-fA-F0-9]+>\]', message):
            return 0.95
        if re.search(r'exited abnormally', msg_lower):
            return 0.9
        if 'shutdown failed' in msg_lower:
            return 0.95
        if msg_lower.strip().endswith('failed'):
            return 0.9

        # Fault word scoring
        fault_score = 0
        for severity, words in self.fault_words.items():
            weight = {'critical': 1.0, 'high': 0.8, 'medium': 0.5}[severity]
            for word in words:
                if re.search(rf'\b{word}\b', msg_lower):
                    fault_score += weight

        # Technical pattern matching
        for pattern in self.tech_patterns:
            if re.search(pattern, msg_lower, re.IGNORECASE):
                fault_score += 0.8

        return min(1.0, fault_score / 1.5)

    def classify_fault_type(self, message, is_fault):
        """Determine specific fault type"""
        if not is_fault:
            return 'normal'

        msg_lower = message.lower()

        if re.search(r'exited abnormally|terminated abnormally|abnormal.*termination', msg_lower):
            return 'process_fault'
        elif re.search(r'\[<[a-fA-F0-9]+>\]|.*\+0x[a-fA-F0-9]+', message):
            return 'kernel_fault'
        elif any(word in msg_lower for word in ['memory', 'oom', 'vm:', 'out of memory', 'mm_fault', 'segfault']):
            return 'memory_fault'
        elif any(word in msg_lower for word in ['panic', 'crash', 'kernel', 'oops', 'do_exit', 'page_fault']):
            return 'kernel_fault'
        elif any(word in msg_lower for word in ['auth', 'login', 'failure', 'denied', 'unauthorized']):
            return 'security_fault'
        elif any(word in msg_lower for word in ['shutdown', 'stop', 'service', 'daemon']):
            return 'service_fault'
        elif any(word in msg_lower for word in ['hardware', 'disk', 'thermal', 'machine check']):
            return 'hardware_fault'
        else:
            return 'general_fault'

    def parse_log_entry(self, log_line, current_year=2024):
        """Parse single log entry"""
        log_line = log_line.strip()
        if not log_line:
            return None

        match = self.log_pattern.match(log_line)

        if not match:
            return {
                'timestamp': None,
                'service': 'unknown',
                'message': log_line
            }

        timestamp_str, hostname, service, message = match.groups()

        try:
            timestamp = datetime.strptime(f"{current_year} {timestamp_str}", "%Y %b %d %H:%M:%S")
        except ValueError:
            timestamp = None

        return {
            'timestamp': timestamp,
            'service': self.normalize_service_name(service.strip()),
            'message': message.strip()
        }

    def process_realtime_batch(self, realtime_logs, current_year=2024):
        """
        Process batch of real-time logs and print results directly
        Input: List of log strings
        Output: Prints classification results in tabular format
        """
        print("🚀 Processing real-time log batch...")
        print("=" * 120)

        if not realtime_logs:
            print("❌ No logs to process")
            return

        # Parse all logs
        parsed_logs = []
        for log_line in realtime_logs:
            parsed_entry = self.parse_log_entry(log_line, current_year)
            if parsed_entry:
                parsed_logs.append(parsed_entry)

        if not parsed_logs:
            print("❌ No valid logs found")
            return

        # Fast clustering for small batches
        messages = [log['message'] for log in parsed_logs]
        normalized_messages = [self.normalize_message(msg) for msg in messages]

        # Simple clustering for real-time processing
        cluster_ids = self._fast_clustering(normalized_messages)

        # Process each log
        results = []
        for i, parsed_log in enumerate(parsed_logs):
            fault_prob = self.analyze_fault_probability(parsed_log['message'])
            is_fault = fault_prob > 0.5
            fault_type = self.classify_fault_type(parsed_log['message'], is_fault)
            normalized_msg = self.normalize_message(parsed_log['message'])

            result = {
                'timestamp': parsed_log['timestamp'].strftime('%Y-%m-%d %H:%M:%S') if parsed_log['timestamp'] else 'Unknown',
                'service': parsed_log['service'],
                'message': parsed_log['message'],
                'normalized_message': normalized_msg,
                'is_fault': is_fault,
                'fault_type': fault_type,
                'cluster_id': cluster_ids[i] if i < len(cluster_ids) else 0
            }
            results.append(result)

        # Print results in table format
        self._print_results_table(results)

        # Print summary
        self._print_summary(results)

        return results

    def _fast_clustering(self, normalized_messages):
        """Fast clustering for real-time processing"""
        if len(normalized_messages) < 3:
            return [0] * len(normalized_messages)

        try:
            vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
            features = vectorizer.fit_transform(normalized_messages)
            n_clusters = min(3, max(2, len(normalized_messages) // 5))
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=3)
            return kmeans.fit_predict(features).tolist()
        except:
            return [0] * len(normalized_messages)

    def _print_results_table(self, results):
        """Print results in formatted table"""
        print(f"📊 REAL-TIME LOG CLASSIFICATION RESULTS")
        print("=" * 120)

        # Header
        print(f"{'#':<3} {'FAULT':<6} {'TYPE':<15} {'SERVICE':<12} {'TIMESTAMP':<19} {'MESSAGE':<50}")
        print("-" * 120)

        # Results
        for i, result in enumerate(results, 1):
            fault_status = "🚨 YES" if result['is_fault'] else "✅ NO "
            fault_type = result['fault_type'][:14]
            service = result['service'][:11]
            timestamp = result['timestamp'][:18]
            message = result['message'][:49] + ("..." if len(result['message']) > 49 else "")

            print(f"{i:<3} {fault_status:<6} {fault_type:<15} {service:<12} {timestamp:<19} {message:<50}")

    def _print_summary(self, results):
        """Print processing summary"""
        total = len(results)
        faults = sum(1 for r in results if r['is_fault'])

        print("\n" + "=" * 120)
        print(f"📈 BATCH PROCESSING SUMMARY")
        print("=" * 50)
        print(f"📊 Total logs processed: {total}")
        print(f"🚨 Faults detected: {faults} ({faults/total*100:.1f}%)")
        print(f"✅ Normal operations: {total-faults} ({(total-faults)/total*100:.1f}%)")

        if faults > 0:
            fault_types = {}
            for result in results:
                if result['is_fault']:
                    fault_type = result['fault_type']
                    fault_types[fault_type] = fault_types.get(fault_type, 0) + 1

            print(f"\n🔍 FAULT TYPE BREAKDOWN:")
            for fault_type, count in fault_types.items():
                print(f"  {fault_type}: {count}")

        print("=" * 50)

# ============================================================================
# USAGE EXAMPLE
# ============================================================================

def process_realtime_logs(realtime_logs):
    """
    Main function to process real-time logs
    Input: List of log strings
    Output: Prints classification results directly
    """
    processor = RealTimeLogProcessor()
    return processor.process_realtime_batch(realtime_logs)

# Example usage with sample real-time logs
if __name__ == "__main__":
    # Sample real-time logs (replace with your actual real-time log input)
    sample_realtime_logs = [
        "Jun  9 06:06:21 combo kernel: kernel panic - not syncing: Attempted to kill init!",
        "Jun  9 06:06:22 combo sshd[1234]: authentication failure; user=root",
        "Jun  9 06:06:23 combo httpd[5678]: server started successfully",
        "Jun  9 06:06:24 combo kernel: [<c011f8b2>] panic+0x1a/0x120",
        "Jun  9 06:06:25 combo mysqld[9999]: ALERT exited abnormally with [1]",
        "Jun  9 06:06:26 combo ftpd[2345]: connection from 192.168.1.100",
        "Jun  9 06:06:27 combo kernel: Out of Memory: Killed process 12345 (nginx)."
    ]

    # Process the real-time logs
    results = process_realtime_logs(sample_realtime_logs)


🚀 Processing real-time log batch...
📊 REAL-TIME LOG CLASSIFICATION RESULTS
#   FAULT  TYPE            SERVICE      TIMESTAMP           MESSAGE                                           
------------------------------------------------------------------------------------------------------------------------
1   🚨 YES  kernel_fault    kernel       2024-06-09 06:06:2  kernel panic - not syncing: Attempted to kill ini...
2   🚨 YES  security_fault  sshd         2024-06-09 06:06:2  authentication failure; user=root                 
3   ✅ NO   normal          httpd        2024-06-09 06:06:2  server started successfully                       
4   🚨 YES  kernel_fault    kernel       2024-06-09 06:06:2  [<c011f8b2>] panic+0x1a/0x120                     
5   🚨 YES  process_fault   mysqld       2024-06-09 06:06:2  ALERT exited abnormally with [1]                  
6   ✅ NO   normal          ftpd         2024-06-09 06:06:2  connection from 192.168.1.100                     
7   🚨 YES  memory_fault  